# MOTEL Record Builder

This notebook creates, validates, and exports **technology parameter records** according to the schema defined in `data_structure.yaml`.

## Workflow
1. Load and inspect the schema
2. Fill in record fields using Python dicts or helper functions
3. Validate using Pydantic models
4. Export records to JSON (or CSV)

### Schema layers
| Layer | Entities |
|---|---|
| **Main** | `record` |
| **Secondary** | `tech`, `review`, `source` |
| **Supplementary** | `contributor`, `process` |
| **Controlled vocabulary** | `attribute`, `carrier`, `geographic_scope`, `temporal_scope`, `capacity_scope`, `system_boundary`, `uncertainty` |

In [17]:
export_data = False
clear_datasets = False

In [4]:
import importlib
import json
from datetime import date
from pathlib import Path

import pandas as pd

import motel_record_builder as mrb

mrb = importlib.reload(mrb)
mp = mrb.Motel_Platform()

SCHEMA_PATH = Path("data_structure.yaml")

print("Motel_Platform instance ready.")

Motel_Platform instance ready.


In [5]:
# ---------------------------------------------------------------------------
# Load schema entities from helper module
# ---------------------------------------------------------------------------
sections = mp.parse_schema_entities(SCHEMA_PATH)

print("Schema entities loaded:")
for entity, layer in sections.items():
    print(f"  [{layer:15s}]  {entity}")

Schema entities loaded:
  [records        ]  record
  [secondary      ]  tech
  [secondary      ]  review
  [secondary      ]  source
  [supplementary  ]  contributor
  [supplementary  ]  process
  [predefined     ]  attribute
  [predefined     ]  carrier
  [predefined     ]  geographic_scope
  [predefined     ]  temporal_scope
  [predefined     ]  capacity_scope
  [predefined     ]  system_boundary
  [predefined     ]  uncertainty


# Adding datat step by step

## step 1: show template and user provide draft input

In [18]:
# Step 1: show template and let user provide draft input
DATASETS_DIR = Path("datasets")
DATASETS_DIR.mkdir(parents=True, exist_ok=True)

if clear_datasets:
    cleared_files = mp.clear_dataset_data(DATASETS_DIR, schema_path=SCHEMA_PATH)
    print(f"Cleared dataset files: {len(cleared_files)}")

# Create template files only on first run when no CSVs exist yet.
if not any(DATASETS_DIR.rglob("*.csv")):
    mp.export_dataset_csv_templates(SCHEMA_PATH, DATASETS_DIR)

data_store = mp.load_data_store(DATASETS_DIR)

print("Loaded dataset object summary:")
print(f"  tech: {len(data_store.tech)}")
print(f"  source: {len(data_store.source)}")
print(f"  contributor: {len(data_store.contributor)}")
print(f"  process: {len(data_store.process)}")
print(f"  geographic_scope: {len(data_store.geographic_scope)}")
print(f"  temporal_scope: {len(data_store.temporal_scope)}")
print(f"  capacity_scope: {len(data_store.capacity_scope)}")
print(f"  system_boundary: {len(data_store.system_boundary)}")

record_template = {
    "tech_id": "TECH_*",
    "source_id": "SRC_*",
    "geographic_scope": "GEO_*",
    "temporal_scope": "TIME_*",
    "capacity_scope": "CAP_*",
    "system_boundary": "COND_*",
    "values": [
        {
            "attribute_id": "ATTR_*",
            "value": "<number/text/...>",
            "value_format": "int|float|controlled_vocab",
            "value_type": "numeric|text|boolean|array|timeseries|distribution",
            "unit": "<unit>",
            "uncertainty": "<float or {min,max,...}>",
            "note": "<optional>",
        }
    ],
}

record_draft = {
    "tech_id": "TECH_SOLAR_PV_UTILITY",
    "source_id": "SRC_NREL_ATB_2024",
    "geographic_scope": "GEO_USA",
    "temporal_scope": "TIME_2030",
    "capacity_scope": "CAP_UTILITY",
    "system_boundary": "COND_ISO_BASELOAD",
    "values": [
        {
            "attribute_id": "ATTR_CAPEX",
            "value": 920.0,
            "value_format": "float",
            "value_type": "numeric",
            "unit": "USD/kW",
            "uncertainty": {"min": 780.0, "max": 1100.0},
            "note": "Utility-scale, moderate resource, 2030 projection",
        },
        {
            "attribute_id": "ATTR_EFFICIENCY",
            "value": 0.21,
            "value_format": "float",
            "value_type": "numeric",
            "unit": "fraction",
            "uncertainty": 0.01,
            "note": "Module efficiency (DC), monocrystalline silicon",
        },
        {
            "attribute_id": "ATTR_LIFETIME",
            "value": 30,
            "value_format": "int",
            "value_type": "numeric",
            "unit": "years",
            "uncertainty": None,
            "note": None,
        },
    ],
}

print("Template and draft prepared.")
print("Template:")
print(json.dumps(record_template, indent=2))
print("Draft:")
print(json.dumps(record_draft, indent=2))

Loaded dataset object summary:
  tech: 0
  source: 0
  contributor: 0
  process: 0
  geographic_scope: 0
  temporal_scope: 0
  capacity_scope: 0
  system_boundary: 0
Template and draft prepared.
Template:
{
  "tech_id": "TECH_*",
  "source_id": "SRC_*",
  "geographic_scope": "GEO_*",
  "temporal_scope": "TIME_*",
  "capacity_scope": "CAP_*",
  "system_boundary": "COND_*",
  "values": [
    {
      "attribute_id": "ATTR_*",
      "value": "<number/text/...>",
      "value_format": "int|float|controlled_vocab",
      "value_type": "numeric|text|boolean|array|timeseries|distribution",
      "unit": "<unit>",
      "uncertainty": "<float or {min,max,...}>",
      "note": "<optional>"
    }
  ]
}
Draft:
{
  "tech_id": "TECH_SOLAR_PV_UTILITY",
  "source_id": "SRC_NREL_ATB_2024",
  "geographic_scope": "GEO_USA",
  "temporal_scope": "TIME_2030",
  "capacity_scope": "CAP_UTILITY",
  "system_boundary": "COND_ISO_BASELOAD",
  "values": [
    {
      "attribute_id": "ATTR_CAPEX",
      "value": 92

In [7]:
# Optional: inspect reFuel source file for additional examples
excel_path = Path("input") / "reFuel_TechDatabase_Clean_2026-05-29.xlsx"
sheet_names, preview_sheet, source_df_raw = mp.inspect_refuel_source_file(excel_path)

print("Sheets:", sheet_names)
print(f"Using preview sheet: {preview_sheet}")
print(f"Rows: {len(source_df_raw)}, Columns: {len(source_df_raw.columns)}")
print("Columns:")
for c in source_df_raw.columns:
    print(" -", c)

source_df_raw.head(5)

Sheets: ['Metadata', 'Nomenclature', 'Carrier', 'ConvTech', 'StorTech', 'EmbeddedCarbon', 'Reference']
Using preview sheet: Metadata
Rows: 62, Columns: 10
Columns:
 - Col #
 - Applies To
 - Column Group
 - Column Header
 - Variable Name
 - Data Type
 - Unit / Format
 - Allowed Values
 - Description
 - Note


,Col #,Applies To,Column Group,Column Header,Variable Name,Data Type,Unit / Format,Allowed Values,Description,Note
0,NaN,NaN,CLASSIFICATION,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1,Both,Classification,Technology ID,tech_id,string,—,See ConvTech / StorTech,Unique technology identifier following {Input}...,Primary key (with tech_year)
2,2,Both,Classification,Technology Year,tech_year,integer,[year],"{2025, 2030, 2040, 2050}",Projection year for which the techno-economic ...,Time-invariant techs use 2025
3,3,Both,Classification,Technology Class,technology_class,categorical,—,See Nomenclature 4.1,High-level technology classification for filte...,NaN
4,4,Both,Classification,Description,description,string,—,Free text,Brief textual description of the technology co...,NaN


In [8]:
# Inspect candidate rows from ConvTech and StorTech for example generation
conv_df = pd.read_excel(excel_path, sheet_name="ConvTech")
stor_df = pd.read_excel(excel_path, sheet_name="StorTech")

print("ConvTech columns (first 25):")
print(conv_df.columns.tolist()[:25])
print("\nStorTech columns (first 25):")
print(stor_df.columns.tolist()[:25])

print("\nConvTech sample rows:")
display(conv_df.head(3))
print("\nStorTech sample rows:")
display(stor_df.head(3))

ConvTech columns (first 25):
['Classification', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Unnamed: 6', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Unnamed: 12', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Unnamed: 20', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24']

StorTech columns (first 25):
['Classification', 'Unnamed: 1', 'Unnamed: 2', 'Unnamed: 3', 'Unnamed: 4', 'Unnamed: 5', 'Attributes', 'Unnamed: 7', 'Unnamed: 8', 'Unnamed: 9', 'Unnamed: 10', 'Unnamed: 11', 'Storage Parameters', 'Unnamed: 13', 'Unnamed: 14', 'Unnamed: 15', 'Unnamed: 16', 'Unnamed: 17', 'Unnamed: 18', 'Unnamed: 19', 'Economics', 'Unnamed: 21', 'Unnamed: 22', 'Unnamed: 23', 'Unnamed: 24']

ConvTech sample rows:


,Classification,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33,Unnamed: 34,Unnamed: 35,Unnamed: 36,Unnamed: 37,Unnamed: 38,Unnamed: 39
0,Technology ID,Technology Year,Technology Class,Description,Unit Operation,Tech Type,Reference Unit Size,Cost Base,Currency,Technology Readiness Level,...,OPEX per Energy,Discount Rate,Value Range Check,Uncertainty Rating,Mass Inflows,Mass Outflows,Energy Balance Error,Mass Balance Error,Balance Pass Flag,Source References
1,tech_id,tech_year,technology_class,description,unit_operation,tech_type,reference_unit_size,cost_base,currency,trl,...,opex_per_energy,discount_rate_pct,value_range_check,uncertainty_rating,mass_inflows,mass_outflows,energy_balance_error_pct,mass_balance_error_pct,balance_pass_flag,list_of_source_id
2,Electricity generation,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN



StorTech sample rows:


,Classification,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Attributes,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 17,Unnamed: 18,Unnamed: 19,Economics,Unnamed: 21,Unnamed: 22,Unnamed: 23,Unnamed: 24,Unnamed: 25,References
0,Technology ID,Technology Year,Technology Class,Description,Unit Operation,Tech Type,Storage Carrier,Cost Base,Currency,TRL,...,Minimum SOC,Maximum SOC,Standby Loss per Hour,Economic Lifetime,CAPEX One-Time,CAPEX per Storage Capacity (kWh),OPEX One-Time,OPEX per Storage Capacity (kWh),Discount Rate,Source References
1,tech_id,tech_year,technology_class,description,unit_operation,tech_type,storage_carrier,cost_base,currency,trl,...,min_soc,max_soc,standby_loss_per_hour,lifetime_yr,capex_one_time,capex_per_stor_capacity,opex_one_time,opex_per_stor_capacity_yr,discount_rate_pct,list_of_source_id
2,ElBdg_Battery_ElBdg,2025,Battery storage,"Electric battery storage at home, 2025",Li-ion battery (residential),Storage,ElBdg,ECA,EUR,9,...,0,0.95,0,20,0,1000,0,25,NaN,"""capex_per_stor_capacity"": web_BatteryCH_2023;..."


In [9]:


# Build 2-3 example record drafts from reFuel workbook data

def normalize_refuel_sheet(df_raw: pd.DataFrame) -> pd.DataFrame:
    # Row 1 contains variable names (tech_id, tech_year, etc.)
    variable_row = df_raw.iloc[1]
    df = df_raw.iloc[2:].copy()
    df.columns = [str(x).strip() for x in variable_row]
    df = df.reset_index(drop=True)

    # Keep rows with actual tech IDs and numeric tech_year (removes section headers)
    if "tech_id" in df.columns:
        df = df[df["tech_id"].notna()]
        df = df[df["tech_id"].astype(str).str.strip() != ""]
    if "tech_year" in df.columns:
        year_num = pd.to_numeric(df["tech_year"], errors="coerce")
        df = df[year_num.notna()].copy()
        df["tech_year"] = year_num[year_num.notna()].astype(int)

    return df


def to_float_or_none(x):
    try:
        if pd.isna(x):
            return None
        return float(x)
    except Exception:
        return None


def build_example_from_row(row: pd.Series, category: str) -> dict:
    tech_id = str(row.get("tech_id", "")).strip()
    source_refs = str(row.get("list_of_source_id", "")).strip()

    lifetime = to_float_or_none(row.get("lifetime_yr"))

    # Efficiency candidates differ between sheets; pick first usable one.
    efficiency = None
    for c in [
        "overall_efficiency",
        "electrical_efficiency",
        "efficiency",
        "eta",
        "roundtrip_efficiency",
        "rte",
    ]:
        if c in row.index:
            efficiency = to_float_or_none(row.get(c))
            if efficiency is not None:
                break

    # CAPEX candidates differ between conversion and storage sheets.
    capex = None
    for c in [
        "capex_per_capacity",
        "capex_per_stor_capacity",
        "capex_per_storage_capacity",
        "capex_one_time",
    ]:
        if c in row.index:
            capex = to_float_or_none(row.get(c))
            if capex is not None:
                break

    note_suffix = f" Source refs: {source_refs}" if source_refs and source_refs != "nan" else ""

    values = []
    if capex is not None:
        values.append(
            {
                "attribute_id": "ATTR_CAPEX",
                "value": capex,
                "value_format": "float",
                "value_type": "numeric",
                "unit": "EUR/kW",
                "uncertainty": None,
                "note": f"Derived from reFuel CAPEX field for {tech_id}.{note_suffix}".strip(),
            }
        )
    if efficiency is not None:
        values.append(
            {
                "attribute_id": "ATTR_EFFICIENCY",
                "value": efficiency,
                "value_format": "float",
                "value_type": "numeric",
                "unit": "fraction",
                "uncertainty": None,
                "note": f"Derived from reFuel efficiency field for {tech_id}.{note_suffix}".strip(),
            }
        )
    if lifetime is not None:
        values.append(
            {
                "attribute_id": "ATTR_LIFETIME",
                "value": int(lifetime) if float(lifetime).is_integer() else lifetime,
                "value_format": "int" if float(lifetime).is_integer() else "float",
                "value_type": "numeric",
                "unit": "years",
                "uncertainty": None,
                "note": f"Derived from reFuel lifetime_yr for {tech_id}.{note_suffix}".strip(),
            }
        )

    return {
        "tech_id": tech_id,
        "source_id": "SRC_REFUEL_2026",
        "geographic_scope": "GEO_CH",
        "temporal_scope": f"TIME_{int(row.get('tech_year', 2025))}",
        "capacity_scope": "CAP_UTILITY" if category == "conversion" else "CAP_BUILDING",
        "system_boundary": "COND_ISO_BASELOAD",
        "values": values,
    }


conv_clean = normalize_refuel_sheet(conv_df)
stor_clean = normalize_refuel_sheet(stor_df)

# Select examples: 2 conversion + 1 storage (or fewer if not available)
conv_examples = conv_clean.head(2)
stor_examples = stor_clean.head(1)

record_draft_examples = []
for _, r in conv_examples.iterrows():
    record_draft_examples.append(build_example_from_row(r, category="conversion"))
for _, r in stor_examples.iterrows():
    record_draft_examples.append(build_example_from_row(r, category="storage"))

print(f"Generated {len(record_draft_examples)} example draft(s) from reFuel workbook")
for i, ex in enumerate(record_draft_examples, start=1):
    print(f"Example {i}: tech_id={ex['tech_id']}, source_id={ex['source_id']}, time={ex['temporal_scope']}, values={len(ex['values'])}")

record_draft_examples

Generated 3 example draft(s) from reFuel workbook
Example 1: tech_id=NH3_CCGT_El, source_id=SRC_REFUEL_2026, time=TIME_2050, values=2
Example 2: tech_id=NH3_OCGT_El, source_id=SRC_REFUEL_2026, time=TIME_2050, values=2
Example 3: tech_id=ElBdg_Battery_ElBdg, source_id=SRC_REFUEL_2026, time=TIME_2025, values=2


[{'tech_id': 'NH3_CCGT_El',
  'source_id': 'SRC_REFUEL_2026',
  'geographic_scope': 'GEO_CH',
  'temporal_scope': 'TIME_2050',
  'capacity_scope': 'CAP_UTILITY',
  'system_boundary': 'COND_ISO_BASELOAD',
  'values': [{'attribute_id': 'ATTR_CAPEX',
    'value': 1200.0,
    'value_format': 'float',
    'value_type': 'numeric',
    'unit': 'EUR/kW',
    'uncertainty': None,
    'note': 'Derived from reFuel CAPEX field for NH3_CCGT_El. Source refs: "capex_per_capacity", "opex_per_capacity_yr", "technical_efficiency", "opex_per_energy", "lifetime_yr": report_power_to_ammonia_2018; "ratios_in", "ratios_out": conversion_parametrization; "operating_temperature_c": report_danish_energy_2025; "ratios_in", "ratios_out": conversion_parametrization;'},
   {'attribute_id': 'ATTR_LIFETIME',
    'value': 35,
    'value_format': 'int',
    'value_type': 'numeric',
    'unit': 'years',
    'uncertainty': None,
    'note': 'Derived from reFuel lifetime_yr for NH3_CCGT_El. Source refs: "capex_per_capacity

In [10]:
# Optional: run checks for all generated examples
example_check_rows = []
for i, draft in enumerate(record_draft_examples, start=1):
    rep = mp.check_record_dependencies(draft=draft, store=data_store)
    example_check_rows.append(
        {
            "example_no": i,
            "tech_id": draft["tech_id"],
            "source_id": draft["source_id"],
            "temporal_scope": draft["temporal_scope"],
            "capacity_scope": draft["capacity_scope"],
            "ready": rep.ready,
            "missing_count": int((rep.to_dataframe()["exists"] == False).sum()),
        }
    )

example_checks_df = pd.DataFrame(example_check_rows)
example_checks_df

,example_no,tech_id,source_id,temporal_scope,capacity_scope,ready,missing_count
0,1,NH3_CCGT_El,SRC_REFUEL_2026,TIME_2050,CAP_UTILITY,False,6
1,2,NH3_OCGT_El,SRC_REFUEL_2026,TIME_2050,CAP_UTILITY,False,6
2,3,ElBdg_Battery_ElBdg,SRC_REFUEL_2026,TIME_2025,CAP_BUILDING,False,6


In [11]:
record_draft_examples[0]

{'tech_id': 'NH3_CCGT_El',
 'source_id': 'SRC_REFUEL_2026',
 'geographic_scope': 'GEO_CH',
 'temporal_scope': 'TIME_2050',
 'capacity_scope': 'CAP_UTILITY',
 'system_boundary': 'COND_ISO_BASELOAD',
 'values': [{'attribute_id': 'ATTR_CAPEX',
   'value': 1200.0,
   'value_format': 'float',
   'value_type': 'numeric',
   'unit': 'EUR/kW',
   'uncertainty': None,
   'note': 'Derived from reFuel CAPEX field for NH3_CCGT_El. Source refs: "capex_per_capacity", "opex_per_capacity_yr", "technical_efficiency", "opex_per_energy", "lifetime_yr": report_power_to_ammonia_2018; "ratios_in", "ratios_out": conversion_parametrization; "operating_temperature_c": report_danish_energy_2025; "ratios_in", "ratios_out": conversion_parametrization;'},
  {'attribute_id': 'ATTR_LIFETIME',
   'value': 35,
   'value_format': 'int',
   'value_type': 'numeric',
   'unit': 'years',
   'uncertainty': None,
   'note': 'Derived from reFuel lifetime_yr for NH3_CCGT_El. Source refs: "capex_per_capacity", "opex_per_capacit

## step 2: show the check/validation results for the proposed input, per required section

required section:
 - Technology / Process
 - Scope: geographic, temporal, capacity, system boundary
 - (Data) Source: reference...

action: for each required input,
     - how much existing data in each dataset (e.g. technology, process...)
     - is the proposed input already in the existing,
     - if not, what are the close existing to the propose one
     - action to the user: change to existing one or add a new one


In [20]:
## show the target record
record_draft = record_draft_examples[0]

record_draft

{'tech_id': 'NH3_CCGT_El',
 'source_id': 'SRC_REFUEL_2026',
 'geographic_scope': 'GEO_CH',
 'temporal_scope': 'TIME_2050',
 'capacity_scope': 'CAP_UTILITY',
 'system_boundary': 'COND_ISO_BASELOAD',
 'values': [{'attribute_id': 'ATTR_CAPEX',
   'value': 1200.0,
   'value_format': 'float',
   'value_type': 'numeric',
   'unit': 'EUR/kW',
   'uncertainty': None,
   'note': 'Derived from reFuel CAPEX field for NH3_CCGT_El. Source refs: "capex_per_capacity", "opex_per_capacity_yr", "technical_efficiency", "opex_per_energy", "lifetime_yr": report_power_to_ammonia_2018; "ratios_in", "ratios_out": conversion_parametrization; "operating_temperature_c": report_danish_energy_2025; "ratios_in", "ratios_out": conversion_parametrization;'},
  {'attribute_id': 'ATTR_LIFETIME',
   'value': 35,
   'value_format': 'int',
   'value_type': 'numeric',
   'unit': 'years',
   'uncertainty': None,
   'note': 'Derived from reFuel lifetime_yr for NH3_CCGT_El. Source refs: "capex_per_capacity", "opex_per_capacit

### 2a: technology and process

In [ ]:
# Step 2a: Technology / Process checks
report = mp.check_record_dependencies(draft=record_draft, store=data_store)
check_df = report.to_dataframe()

tech_process_fields = ["tech_id", "tech.process_id"]
tech_process_df = check_df[check_df["field"].isin(tech_process_fields)].copy()

print(f"tech entries available: {len(data_store.tech)}")
print(f"process entries available: {len(data_store.process)}")
print(f"draft tech_id: {record_draft.get('tech_id')}")



print(f"overall ready status: {report.ready}")

display(tech_process_df)

tech entries available: 0
process entries available: 0
draft tech_id: NH3_CCGT_El
overall ready status: False


,field,value,dataset,exists,suggestion
0,tech_id,NH3_CCGT_El,tech,False,create new tech entry for 'NH3_CCGT_El'


### 2b: scopes

In [13]:
# Step 2b: Scope checks (geographic / temporal / capacity / system boundary)
scope_fields = [
    "geographic_scope",
    "temporal_scope",
    "capacity_scope",
    "system_boundary",
]
scope_df = check_df[check_df["field"].isin(scope_fields)].copy()

scope_catalog = {
    "geographic_scope": sorted(data_store.geographic_scope),
    "temporal_scope": sorted(data_store.temporal_scope),
    "capacity_scope": sorted(data_store.capacity_scope),
    "system_boundary": sorted(data_store.system_boundary),
}

for scope_name, values in scope_catalog.items():
    draft_value = record_draft.get(scope_name, "")
    print(f"\n{scope_name}: {len(values)} existing value(s)")
    if values:
        exact = draft_value in values
        close = [v for v in values if draft_value and draft_value[:4] in v]
        print(f"  draft value: {draft_value}")
        print(f"  exact match: {exact}")
        print(f"  close existing: {close[:5] if close else 'none'}")
    else:
        print(f"  no existing values; '{draft_value}' would need to be added")

display(scope_df)


geographic_scope: 0 existing value(s)
  no existing values; 'GEO_USA' would need to be added

temporal_scope: 0 existing value(s)
  no existing values; 'TIME_2030' would need to be added

capacity_scope: 0 existing value(s)
  no existing values; 'CAP_UTILITY' would need to be added

system_boundary: 0 existing value(s)
  no existing values; 'COND_ISO_BASELOAD' would need to be added


,field,value,dataset,exists,suggestion
2,geographic_scope,GEO_USA,geographic_scope,False,create new geographic_scope entry for 'GEO_USA'
3,temporal_scope,TIME_2030,temporal_scope,False,create new temporal_scope entry for 'TIME_2030'
4,capacity_scope,CAP_UTILITY,capacity_scope,False,create new capacity_scope entry for 'CAP_UTILITY'
5,system_boundary,COND_ISO_BASELOAD,system_boundary,False,create new system_boundary entry for 'COND_ISO...


### 2c: source

In [14]:
# Step 2c: Source checks
source_df = check_df[check_df["field"] == "source_id"].copy()

print(f"source entries available: {len(data_store.source)}")
print(f"draft source_id: {record_draft.get('source_id')}")

display(source_df)

print(f"Ready to add record? {report.ready}")

source entries available: 0
draft source_id: SRC_NREL_ATB_2024


,field,value,dataset,exists,suggestion
1,source_id,SRC_NREL_ATB_2024,source,False,create new source entry for 'SRC_NREL_ATB_2024'


Ready to add record? False


## step 3: review if all the section is checked (select existing or add a new now)

In [15]:
# Step 3: review status and decide (use existing vs add new)
missing_df = check_df[check_df["exists"] == False]

if missing_df.empty:
    print("All required sections are satisfied. Ready to add the record.")
else:
    print("Some required sections are missing. Review and choose an action:")
    display(missing_df[["field", "dataset", "value", "suggestion"]])

all_checks_passed = missing_df.empty
print(f"all_checks_passed = {all_checks_passed}")

Some required sections are missing. Review and choose an action:


,field,dataset,value,suggestion
0,tech_id,tech,TECH_SOLAR_PV_UTILITY,create new tech entry for 'TECH_SOLAR_PV_UTILITY'
1,source_id,source,SRC_NREL_ATB_2024,create new source entry for 'SRC_NREL_ATB_2024'
2,geographic_scope,geographic_scope,GEO_USA,create new geographic_scope entry for 'GEO_USA'
3,temporal_scope,temporal_scope,TIME_2030,create new temporal_scope entry for 'TIME_2030'
4,capacity_scope,capacity_scope,CAP_UTILITY,create new capacity_scope entry for 'CAP_UTILITY'
5,system_boundary,system_boundary,COND_ISO_BASELOAD,create new system_boundary entry for 'COND_ISO...


all_checks_passed = False


## step 4: add the record to the data; the validation the data again

In [16]:
if export_data:

    # Step 4: add record and validate again
    OUTPUT_DIR = Path("output")
    OUTPUT_DIR.mkdir(exist_ok=True)

    # Keep datasets in the dedicated datasets folder.
    if "DATASETS_DIR" not in globals():
        DATASETS_DIR = Path("datasets")
    DATASETS_DIR.mkdir(parents=True, exist_ok=True)

    # Optional helper branch for this example: auto-create any missing required entities.
    if not all_checks_passed:
        print("Auto-creating missing required entries for this demo...")

        mp.register_process(
            data_store,
            mp.Process(
                process_id="PROC_SOLAR_PV_UTILITY",
                process_name="Solar PV Utility Conversion",
                process_description="Utility-scale solar PV conversion process",
                process_type="conversion",
                process_category="renewable",
            ),
        )

        mp.register_tech(
            data_store,
            mp.Tech(
                tech_id="TECH_SOLAR_PV_UTILITY",
                technology_name="Solar PV Utility-Scale",
                ehubx_tech_id="solar_pv_util",
                ontology_iri="https://openenergy-platform.org/ontology/oeo/OEO_00010013",
                process_id="PROC_SOLAR_PV_UTILITY",
            ),
        )

        mp.register_source(
            data_store,
            mp.Source(
                source_id="SRC_NREL_ATB_2024",
                source_description="NREL Annual Technology Baseline 2024",
                source_type="database",
                link="https://atb.nrel.gov/",
                access_date=date(2024, 6, 1),
                confidence_level="high",
                assessment_method="modeled",
                assessment_date=date.today(),
            ),
        )

        mp.register_scope_value(data_store, "geographic_scope", "GEO_USA")
        mp.register_scope_value(data_store, "temporal_scope", "TIME_2030")
        mp.register_scope_value(data_store, "capacity_scope", "CAP_UTILITY")
        mp.register_scope_value(data_store, "system_boundary", "COND_ISO_BASELOAD")

    # Final check before creation
    final_report = mp.check_record_dependencies(draft=record_draft, store=data_store)
    print(f"Final ready status: {final_report.ready}")
    display(final_report.to_dataframe())

    solar_record = mp.create_record_if_ready(draft=record_draft, store=data_store)
    solar_record.display()

    written_dataset_files = mp.export_data_store_csv(data_store, DATASETS_DIR)
    all_records = data_store.records
    jsonl_path = mp.export_records_jsonl(all_records, OUTPUT_DIR / "records.jsonl")
    csv_path = mp.export_records_csv(all_records, OUTPUT_DIR / "records.csv")

    print(f"Exported JSONL: {jsonl_path}")
    print(f"Exported records CSV: {csv_path}")
    print("Updated dataset files:")
    for p in written_dataset_files:
        print(f"  - {p}")

    df = mp.records_to_dataframe(all_records)
    df